In [11]:
from click import prompt
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
TRANSCRIPT_PATH = 'data/Transcripts/GOOGL/2019-Feb-04-GOOGL.txt'
from speech_parser.transcript_parser import parse_transcript_to_json
parse_transcript_to_json(TRANSCRIPT_PATH)

WindowsPath('data/processed/2019-Feb-04-GOOGL.json')

In [13]:
import json
import pandas as pd
from pathlib import Path
from nltk.tokenize import sent_tokenize
from tqdm.auto import tqdm

# Construct the output JSON file path
json_file_path = Path(TRANSCRIPT_PATH)
json_output_path = Path('data/processed') / (json_file_path.stem + '.json')

# Load the JSON file
with open(json_output_path, 'r') as f:
    transcript_data = json.load(f)

# Extract all speech strings from presentation and qa fields
all_speeches = []

# Get speeches from presentation field
if 'presentation' in transcript_data:
    for speaker_entry in transcript_data['presentation']:
        if 'Speech' in speaker_entry and speaker_entry['Speech']:
            all_speeches.append(speaker_entry['Speech'])

# Get speeches from qa field
if 'qa' in transcript_data:
    for speaker_entry in transcript_data['qa']:
        if 'Speech' in speaker_entry and speaker_entry['Speech']:
            all_speeches.append(speaker_entry['Speech'])

# Tokenize all speeches into sentences
all_sentences = []
for speech in all_speeches:
    sentences = sent_tokenize(speech)
    all_sentences.extend(sentences)

# Create DataFrame with tokenized sentences
df_speeches = pd.DataFrame({
    'sentence': all_sentences
})

print(f"Total speeches extracted: {len(all_speeches)}")
print(f"Total sentences after tokenization: {len(all_sentences)}")
print(f"\nDataFrame shape: {df_speeches.shape}")
print(f"\nFirst 5 sentences:")
df_speeches.head()

Total speeches extracted: 54
Total sentences after tokenization: 461

DataFrame shape: (461, 1)

First 5 sentences:


C:\Users\aydin\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,sentence
0,"Good day, ladies and gentlemen, and welcome to..."
1,(Operator Instructions) I'd now like to turn t...
2,Please go ahead.
3,Thank you.
4,"Good afternoon, everyone, and welcome to Alpha..."


In [14]:
from dotenv import load_dotenv
import anthropic
import os
load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [15]:
def get_hedging_on_sentence(sentence):
    """
    Returns an LLM prompt designed for the LLM to rank sentence as hedging or not.
    :param sentence: sentence from a transcript as a sentence
    :return: binary outcome (1 = hedging, 0 = not hedging)
    """
    PROMPT = f"""You are a financial language expert specialising in earnings call analysis.

    Label the following sentence as hedging (1) or not hedging (0).

    Hedging language includes any of the following:
    - Modal verbs expressing uncertainty: may, might, could, would, should
    - Epistemic verbs: believe, think, expect, anticipate, feel, assume
    - Approximators: approximately, around, roughly, about
    - Probability expressions: likely, unlikely, possibly, probably, perhaps
    - Conditional framing: assuming, subject to, if, provided that
    - Attribution hedges: based on current trends, according to our estimates
    - Distancing language: it appears, it seems, there is reason to believe
    - Negative hedges: cannot guarantee, no assurance, cannot predict

    Important:
    - Not every modal verb is hedging. "We will pay the dividend"
    is NOT hedging. "We believe we will pay the dividend" IS hedging.
    Direct factual statements about past results are NOT hedging.

    Sentences reporting figures or statistics using words like "estimated," "approximately," or "around" ARE hedging — the speaker is deliberately qualifying the precision of the claim.

    Sentence: "{sentence}"

    Reply with only 0 or 1. No explanation."""

    return PROMPT

def add_hedging_labels(
    sentences_df,
    model="claude-haiku-4-5-20251001",
    output_csv_path="labelled_sentences.csv",
    save_every=50,
):
    """
    Label each sentence in `sentences_df` as hedging (1) or not hedging (0).
    """
    if not isinstance(sentences_df, pd.DataFrame):
        raise TypeError("sentences_df must be a pandas DataFrame")
    if "sentence" not in sentences_df.columns:
        raise ValueError("sentences_df must include a 'sentence' column")
    if not isinstance(save_every, int) or save_every <= 0:
        raise ValueError("save_every must be a positive integer")

    sentences_with_labels = sentences_df.copy()
    labels = []
    checkpoint_buffer = []
    output_path = Path(output_csv_path)
    is_first_write = True
    sentence_values = sentences_with_labels["sentence"].tolist()

    for idx, sentence in enumerate(
        tqdm(
            sentence_values,
            total=len(sentence_values),
            desc="Labelling hedging sentences",
            unit="sentence",
        ),
        start=1,
    ):
        sentence_text = "" if pd.isna(sentence) else str(sentence)
        response = client.messages.create(
            model=model,
            max_tokens=5,
            messages=[{"role": "user", "content": get_hedging_on_sentence(sentence_text)}],
        )
        label_text = response.content[0].text.strip()
        if label_text not in {"0", "1"}:
            raise ValueError(f"Unexpected label returned by model: {label_text!r}")
        label_value = int(label_text)
        labels.append(label_value)

        checkpoint_buffer.append({"sentence": sentence_text, "isHedge": label_value})
        if idx % save_every == 0:
            pd.DataFrame(checkpoint_buffer).to_csv(
                output_path,
                mode="w" if is_first_write else "a",
                header=is_first_write,
                index=False,
            )
            checkpoint_buffer.clear()
            is_first_write = False

    if checkpoint_buffer:
        pd.DataFrame(checkpoint_buffer).to_csv(
            output_path,
            mode="w" if is_first_write else "a",
            header=is_first_write,
            index=False,
        )

    sentences_with_labels["isHedge"] = labels
    return sentences_with_labels

sentences_with_labels = add_hedging_labels(df_speeches)
sentences_with_labels.head()

Labelling hedging sentences: 100%|██████████| 461/461 [08:14<00:00,  1.07s/sentence]


,sentence,isHedge
0,"Good day, ladies and gentlemen, and welcome to...",0
1,(Operator Instructions) I'd now like to turn t...,0
2,Please go ahead.,0
3,Thank you.,0
4,"Good afternoon, everyone, and welcome to Alpha...",0
